In [1]:
import os
import sys
import numpy as np
import tensorflow as tf
from pathlib import Path
from tqdm import tqdm

# Add project root to path and fix working directory
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

os.chdir(project_root)

from src.data.loader import GliomaDataLoader
from src.data.augmentation import AugmentedDataGenerator
from src.models.cvae import CVAE

# GPU Check
print("TensorFlow Version:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())

c:\Users\Andrei\anaconda3\envs\glioma\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.6 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


TensorFlow Version: 2.20.0
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [2]:
# Setup experiment from config
from src.utils.config_manager import ConfigManager
from src.notebooks.experiment_utils import setup_experiment

config_name = 'exp_108'
config, output_dir = setup_experiment(config_name)

# Load paths from config
DATA_PATH = config['paths']['processed_file']
SPLIT_FILE = config['paths']['splits_file']
SYNTHETIC_OUT_DIR = config['paths']['synthetic_dir']
os.makedirs(SYNTHETIC_OUT_DIR, exist_ok=True)

print(f"✓ Config loaded: {config_name}")
print(f"Data path: {DATA_PATH}")
print(f"Synthetic output: {SYNTHETIC_OUT_DIR}")
print("\nInitializing Loader...")

loader = GliomaDataLoader.from_config(
    h5_data_path=DATA_PATH,
    config=config,
    memory_mapped=False
)

print(f"✓ Loader initialized. Total spectra: {loader.X.shape[0]}")


EXPERIMENT SETUP

Working directory: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data

 Loaded configuration: exp_108
   Description: TSAR Regimen: Fingerprint crop (700-1800), patient cap (500), and dynamic Raman augmentation

 Output directory: C:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\results\results
  Experiment already registered (re-running)

--------------------------------------------------------------------------------
Configuration Summary:
--------------------------------------------------------------------------------
  Preprocessing:
    - AirPLS lambda: 100
    - Signal length: 1738
    - Crop region: [700, 1800]

  CV Strategy:
    - Split source: data/splits/strict_splits.json
    - Use group-aware CV: True
    - Hold out test set: True

  Models:
    - RF n_estimators: 100
    - SVM C: 1.0
    - CVAE epochs: 220


✓ Config loaded: exp_108
Data path: results/exp_102/data/glioma_dataset.h5
Synthetic output: results/exp_108/synthetic

Initializing Lo

In [3]:
# Build CVAE config from centralized exp_103.yaml
cvae_cfg = config['models']['cvae']

# Compute input_shape from actual loaded data (handles cropping)
actual_n_features = loader.X.shape[1]

# Get number of classes from config (binary or 6-class)
num_classes_config = config['models']['cnn']['num_classes']

cvae_config = {
    'input_shape': (actual_n_features, 1),
    'num_classes': num_classes_config,
    'latent_dim': cvae_cfg['latent_dim'],
    'loss_weights': {
        'beta': cvae_cfg['loss_weights']['beta'],
        'alpha': cvae_cfg['loss_weights']['alpha']
    },
    'encoder': {
        'filters': cvae_cfg['encoder']['filters'],
        'kernels': cvae_cfg['encoder']['kernels'],
        'strides': cvae_cfg['encoder']['strides'],
        'use_global_pooling': False,
        'use_skip_gap': False
    },
    'decoder': {
        'initial_filters': cvae_cfg['decoder']['initial_filters'],
        'use_dense_projection': True,
        'filters': cvae_cfg['decoder']['filters'],
        'kernels': cvae_cfg['decoder']['kernels'],
        'strides': cvae_cfg['decoder']['strides']
    }
}
anneal_cfg = cvae_cfg.get('annealing', {'enabled': False})
is_annealing = anneal_cfg.get('enabled', False)

# Training Hyperparameters from config
EPOCHS = cvae_cfg['epochs']
BATCH_SIZE = 1024  # Increased from 256 for CPU efficiency
SAMPLES_PER_CLASS = cvae_cfg['samples_per_class']
LEARNING_RATE = cvae_cfg['learning_rate']
VALIDATION_SPLIT = cvae_cfg['validation_split']

print(f"CVAE Config loaded from {config_name}:")
print(f"  - Epochs: {EPOCHS}")
print(f"  - Batch size: {BATCH_SIZE} (optimized)")
print(f"  - Samples/class: {SAMPLES_PER_CLASS}")
print(f"  - Latent dim: {cvae_cfg['latent_dim']}")
print(f"  - Num classes: {num_classes_config}")
print(f"  - Input shape: {cvae_config['input_shape']} (computed from loaded data)")
if num_classes_config == 2:
    print(f"  - Binary mode: Mutant (0) vs Wildtype (1), generating {SAMPLES_PER_CLASS*2} total synthetic spectra per fold")
else:
    print(f"  - 6-class mode: generating {SAMPLES_PER_CLASS*6} total synthetic spectra per fold")

CVAE Config loaded from exp_108:
  - Epochs: 220
  - Batch size: 1024 (optimized)
  - Samples/class: 20000
  - Latent dim: 32
  - Num classes: 6
  - Input shape: (571, 1) (computed from loaded data)
  - 6-class mode: generating 120000 total synthetic spectra per fold


In [ ]:
# import time
# time.sleep(8*60*60) 

In [4]:
import gc
import pandas as pd

# Extract augmentation config from exp_103
augmentation_config = config.get('augmentation', {})
print(f"\nAugmentation Configuration:")
print(f"  - Enabled: {augmentation_config.get('enabled', False)}")
if augmentation_config.get('enabled'):
    print(f"  - Max shift: {augmentation_config.get('max_shift', 3.0)} pixels")
    print(f"  - Mixup alpha: {augmentation_config.get('mixup_alpha', 0.5)}")

# Get patient cap from config for dynamic per-epoch capping
patient_cap = config.get('cv_strategy', {}).get('patient_cap', 500)
print(f"  - Patient cap (dynamic per epoch): {patient_cap} spectra per patient")

# Setup output directories from config
MODEL_OUT_DIR = config['paths']['models_cvae_dir']
os.makedirs(MODEL_OUT_DIR, exist_ok=True)
os.makedirs(SYNTHETIC_OUT_DIR, exist_ok=True)

print(f"Model weights output: {MODEL_OUT_DIR}")
print(f"Synthetic data output: {SYNTHETIC_OUT_DIR}")
print(f"\n Starting Systematic Synthetic Generation ({SAMPLES_PER_CLASS} per class)")

for fold_data in loader:
    fold_name = fold_data['fold_name']
    fold_history = []
    # Optional: Skip if already generated (useful if it crashes and you restart)
    out_x = f"{SYNTHETIC_OUT_DIR}/{fold_name}_X_synth.npy"
    out_y = f"{SYNTHETIC_OUT_DIR}/{fold_name}_y_synth.npy"
    if os.path.exists(out_x) and os.path.exists(out_y):
        print(f" Skipping {fold_name}, synthetic data already exists.")
        continue

    print(f"\n{'='*60}")
    print(f" Processing {fold_name}")
    print(f"{'='*60}")
    
    # 1. Unpack Data
    X_train, y_train = fold_data['train']
    train_groups = fold_data['train_groups']  # Patient/slide IDs for dynamic capping
    
    # Fix Labels (0-indexed for Keras categorical)
    if np.min(y_train) == 1:
        y_train = y_train - 1
    
    # === Apply binary conversion if exp_105 ===
    if num_classes_config == 2:
        print(f"  Converting to binary: {0,1,2} → Mutant (0), {3,4,5} → Wildtype (1)")
        y_train = np.where(y_train < 3, 0, 1)
    
    # Reshape for CNN/CVAE (handle 2D input)
    if X_train.ndim == 2:
        X_train = X_train.reshape(-1, X_train.shape[1], 1)
    
    # Squeeze to 2D for augmentation generator
    X_train_2d = X_train.squeeze() if X_train.ndim == 3 else X_train
    
    # One-Hot Encode Labels
    y_train_oh = tf.keras.utils.to_categorical(y_train, num_classes=num_classes_config)
    
    # 2. Build & Compile CVAE
    tf.keras.backend.clear_session()
    cvae = CVAE(cvae_config)
    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    cvae.compile(optimizer=optimizer)
    
    # 3. Train CVAE using custom training loop with dynamic per-epoch patient capping + augmentation
    print(f" Training CVAE on {len(X_train)} real samples...")
    print(f"   Epochs={EPOCHS}, Batch={BATCH_SIZE}, Learning_rate={LEARNING_RATE}")
    print(f"   Patient capping: {patient_cap} per patient per epoch (dynamic, freshly sampled)")
    if augmentation_config.get('enabled'):
        print(f"   Augmentation: ENABLED (shift + within-class mixup)")
    else:
        print(f"   Augmentation: DISABLED (unaugmented training)")
    
    
    
    # Define maximums BEFORE the loop
    max_beta = cvae_config['loss_weights'].get('beta', 1.0)
    max_alpha = cvae_config['loss_weights'].get('alpha', 0.2)
    # Manual training loop using custom train_step
    for epoch in range(EPOCHS):
        total_loss = 0.0
        num_batches = 0
        
        if is_annealing:
            # --- DUAL ANNEALING SCHEDULE ---
            
            # Phase 1 & 2: Beta Annealing (e.g., Starts at 10, ends at 90)
            beta_cfg = anneal_cfg.get('beta', {})
            start_beta_ep = beta_cfg.get('start_epoch', 10)
            warmup_beta = beta_cfg.get('warmup_epochs', 80)
            
            if epoch < start_beta_ep:
                cvae.beta = 0.0
            else:
                cvae.beta = min(max_beta, max_beta * ((epoch - start_beta_ep) / warmup_beta))
                
            # Phase 3: Alpha (Classification) Decay (e.g., Starts at 90, ends at 150)
            alpha_cfg = anneal_cfg.get('alpha', {})
            start_alpha_ep = alpha_cfg.get('start_epoch', 90)
            cooldown_alpha = alpha_cfg.get('warmup_epochs', 60) # Note: 'warmup' here means 'cooldown duration'
            
            if epoch < start_alpha_ep:
                cvae.alpha = max_alpha  # Keep classification high during early training
            else:
                # Linearly decrease alpha down to 0
                decay_factor = 1.0 - ((epoch - start_alpha_ep) / cooldown_alpha)
                cvae.alpha = max(0.0, max_alpha * decay_factor)
            # --------------------------------
        else:
            # Old Experiments (101-106) keep constant weights
            cvae.beta = max_beta
            cvae.alpha = max_alpha

        # === DYNAMIC PATIENT CAPPING: Fresh random subsample each epoch ===
        keep_indices = []
        for group_id in np.unique(train_groups):
            mask = (train_groups == group_id)
            indices = np.where(mask)[0]
            
            # Randomly subsample to patient_cap or use all if fewer available
            if len(indices) > patient_cap:
                keep = np.random.choice(indices, size=patient_cap, replace=False)
                keep_indices.extend(keep)
            else:
                keep_indices.extend(indices)
        
        # Sort indices for consistency
        keep_indices = np.sort(np.array(keep_indices))
        
        # Apply dynamic cap to training data for this epoch
        X_train_epoch = X_train_2d[keep_indices]
        y_train_epoch = y_train[keep_indices]
        
        print(f"  Epoch {epoch+1}: Using {len(X_train_epoch)} spectra (capped from {len(X_train_2d)})", end=" ")
        
        # Create augmented dataset generator with capped data
        aug_gen = AugmentedDataGenerator(X_train_epoch, y_train_epoch, augmentation_config)
        train_dataset_epoch = aug_gen.get_tf_dataset(batch_size=BATCH_SIZE, epochs=1, shuffle=True)
        
        # Reshape back to (batch_size, n_features, 1) for CNN/CVAE
        # Use TensorFlow ops (not NumPy methods) for graph-mode tensors
        def reshape_for_cvae(x_batch, y_batch):
            return tf.expand_dims(x_batch, -1), y_batch
        
        train_dataset_epoch = train_dataset_epoch.map(reshape_for_cvae)
        
        for x_batch, y_batch_idx in train_dataset_epoch:
            # Convert index labels to one-hot for CVAE
            y_batch = tf.keras.utils.to_categorical(y_batch_idx.numpy(), num_classes=num_classes_config)
            y_batch = tf.cast(y_batch, tf.float32)  # Convert numpy float64 to TensorFlow float32
            
            # Single train_step call
            losses = cvae.train_step((x_batch, y_batch))
            total_loss += float(cvae.total_loss_tracker.result())
            num_batches += 1
        
        # Print summary only once per epoch
        avg_loss = total_loss / max(num_batches, 1)
        acc = float(cvae.accuracy_tracker.result()) * 100
        
        # Format the line carefully so it stays readable in the console
        print(f"- loss: {avg_loss:.4f}")
        if (epoch + 1) % 1 == 0 or epoch == 0:
            print(f"         [Weights] Beta: {cvae.beta:.3f} | Alpha: {cvae.alpha:.2f}")
            print(f"         [Losses]  MSE: {float(cvae.mse_tracker.result()):.4f} | "
                  f"Cos: {float(cvae.cos_tracker.result()):.4f} | "
                  f"KL: {float(cvae.kl_loss_tracker.result()):.4f} | "
                  f"Class: {float(cvae.class_loss_tracker.result()):.4f}")
            print(f"         [Metrics] Acc: {acc:.3f}%")
        
        epoch_metrics = {
            'epoch': epoch + 1,
            'total_loss': avg_loss,
            'mse_loss': float(cvae.mse_tracker.result()),
            'cos_loss': float(cvae.cos_tracker.result()),
            'kl_loss': float(cvae.kl_loss_tracker.result()),
            'class_loss': float(cvae.class_loss_tracker.result()),
            'accuracy': acc,
            'beta': float(cvae.beta),
            'alpha': float(cvae.alpha)
        }
        fold_history.append(epoch_metrics)
    print(f"[COMPLETE] Training done for {fold_name}")

    history_df = pd.DataFrame(fold_history)
    history_path = f"{MODEL_OUT_DIR}/history_{fold_name}.csv"
    history_df.to_csv(history_path, index=False)
    print(f"[SAVED] Training history: {history_path}")
    
    # 4. Generate Synthetic Data (using trained weights still in memory - no reload needed)
    total_synth_samples = SAMPLES_PER_CLASS * num_classes_config
    print(f"\n[GENERATING] {total_synth_samples} balanced synthetic spectra for {fold_name}...")
    X_synthetic = []
    y_synthetic = []
    
    # Use batching for generation to avoid OOM with large sample counts
    generation_batch_size = min(5000, SAMPLES_PER_CLASS)  # Generate in chunks of 5k or less
    
    for class_idx in range(num_classes_config):
        print(f"   Class {class_idx}: ", end='', flush=True)
        class_samples = []
        
        # Generate in batches
        num_generation_batches = (SAMPLES_PER_CLASS + generation_batch_size - 1) // generation_batch_size
        for batch_i in range(num_generation_batches):
            # Calculate batch size (handle last batch if remainder)
            batch_size = min(generation_batch_size, SAMPLES_PER_CLASS - batch_i * generation_batch_size)
            
            # Create pure noise
            random_latent_vectors = tf.random.normal(shape=(batch_size, cvae_config['latent_dim']))
            
            # Create requested class labels
            requested_classes = tf.convert_to_tensor(
                tf.keras.utils.to_categorical([class_idx] * batch_size, num_classes=num_classes_config),
                dtype=tf.float32
            )
            
            # Decode noise + label into synthetic spectra (using trained decoder in memory)
            generated_spectra = cvae.decoder([random_latent_vectors, requested_classes], training=False)
            class_samples.append(generated_spectra.numpy())
            print(f".", end='', flush=True)
        
        # Concatenate all batches for this class
        X_synthetic.append(np.vstack(class_samples))
        y_synthetic.extend([class_idx] * SAMPLES_PER_CLASS)
        print(f" {SAMPLES_PER_CLASS} samples")
        
    X_synthetic = np.vstack(X_synthetic)
    y_synthetic = np.array(y_synthetic)
    
    # 5. Save CVAE weights and synthetic data
    enc_path = f"{MODEL_OUT_DIR}/encoder_{fold_name}.weights.h5"
    dec_path = f"{MODEL_OUT_DIR}/decoder_{fold_name}.weights.h5"
    
    cvae.encoder.save_weights(enc_path)
    cvae.decoder.save_weights(dec_path)
    
    print(f"[SAVED] Encoder: {enc_path}")
    print(f"[SAVED] Decoder: {dec_path}")
    
    # Save synthetic data
    np.save(out_x, X_synthetic)
    np.save(out_y, y_synthetic)
    print(f"[SAVED] Synthetic data: {out_x}")
    print(f"[SAVED] Synthetic labels: {out_y}")
    print(f"\n[COMPLETE] {fold_name} (training + generation + save)")
    
    # Cleanup for next fold
    del cvae, X_train, X_train_2d, y_train_oh, X_synthetic, y_synthetic, aug_gen, train_dataset_epoch
    tf.keras.backend.clear_session()
    gc.collect()


print("\n All folds complete! Synthetic datasets ready for TSTR evaluation.")

tf.keras.backend.clear_session()
gc.collect()


Augmentation Configuration:
  - Enabled: True
  - Max shift: 3 pixels
  - Mixup alpha: 0.2
  - Patient cap (dynamic per epoch): 500 spectra per patient
Model weights output: results/exp_108/models/cvae
Synthetic data output: results/exp_108/synthetic

 Starting Systematic Synthetic Generation (20000 per class)

 Using CV splits from: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\data\splits\strict_splits.json
    Found 'cv_folds' section with 5 folds
   ! Separate held-out test set exists: 6 samples
   !  These samples will NOT be used in CV training/validation
   Fold 0: 160084 train samples, 45936 val samples
 Skipping fold_0, synthetic data already exists.
   Fold 1: 163393 train samples, 42627 val samples
 Skipping fold_1, synthetic data already exists.
   Fold 2: 167257 train samples, 38763 val samples

 Processing fold_2

 Training CVAE on 167257 real samples...
   Epochs=220, Batch=1024, Learning_rate=0.002
   Patient capping: 500 per patient per epoch (dynamic, fresh

0

In [6]:
%reset -f

## 3. Preprocessing for CVAE
The CVAE expects `(Spectrum, One-Hot-Label)` pairs.

## 5. Diagnostics
Check if the model learned to reconstruct.

## 6. Visual Sanity Check
Reconstruct a random validation sample.

In [11]:
# DEBUG: Print shapes before training loop
print("\n" + "="*60)
print("SHAPE DIAGNOSTICS")
print("="*60)

# Load one fold to check shapes
for fold_data in loader:
    X_train, y_train = fold_data['train']
    
    if X_train.ndim == 2:
        X_train = X_train.reshape(-1, X_train.shape[1], 1)
    
    X_train_2d = X_train.squeeze() if X_train.ndim == 3 else X_train
    
    print(f"X_train_2d shape: {X_train_2d.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"y_train unique values: {np.unique(y_train)}")
    
    aug_gen = AugmentedDataGenerator(X_train_2d, y_train, augmentation_config)
    train_dataset_epoch = aug_gen.get_tf_dataset(batch_size=BATCH_SIZE, epochs=1, shuffle=True)
    train_dataset_epoch = train_dataset_epoch.map(lambda x, y: (tf.expand_dims(x, -1), y))
    
    # Get first batch and inspect
    for x_batch, y_batch_idx in train_dataset_epoch:
        print(f"\nFirst batch shapes:")
        print(f"  x_batch shape: {x_batch.shape}")
        print(f"  y_batch_idx shape: {y_batch_idx.shape}")
        print(f"  y_batch_idx dtype: {y_batch_idx.dtype}")
        print(f"  y_batch_idx sample values: {y_batch_idx[:5].numpy()}")
        
        # Convert to one-hot
        y_batch = tf.keras.utils.to_categorical(y_batch_idx.numpy(), num_classes=6)
        print(f"  y_batch (one-hot) shape: {y_batch.shape}")
        print(f"  y_batch dtype: {y_batch.dtype}")
        
        # Test encoder
        print(f"\nEncoder test:")
        z_mean, z_log_var, z, class_pred = cvae.encoder(x_batch)
        print(f"  z_mean shape: {z_mean.shape}")
        print(f"  z_log_var shape: {z_log_var.shape}")
        print(f"  z shape: {z.shape}")
        print(f"  class_pred shape: {class_pred.shape}")
        
        # Test decoder inputs
        print(f"\nDecoder inputs:")
        print(f"  z shape: {z.shape} (expected: [batch, latent_dim])")
        print(f"  y_batch shape: {y_batch.shape} (expected: [batch, num_classes])")
        
        break
    break

print("="*60)



SHAPE DIAGNOSTICS

 Using CV splits from: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\data\splits\strict_splits.json
    Found 'cv_folds' section with 5 folds
   ! Separate held-out test set exists: 6 samples
   !  These samples will NOT be used in CV training/validation
   Fold 0: 20500 train samples, 5500 val samples
X_train_2d shape: (20500, 571)
y_train shape: (20500,)
y_train unique values: [0 1 2 3 4 5]

First batch shapes:
  x_batch shape: (1024, 571, 1)
  y_batch_idx shape: (1024,)
  y_batch_idx dtype: <dtype: 'int32'>
  y_batch_idx sample values: [4 2 0 4 3]
  y_batch (one-hot) shape: (1024, 6)
  y_batch dtype: float64

Encoder test:
  z_mean shape: (1024, 32)
  z_log_var shape: (1024, 32)
  z shape: (1024, 32)
  class_pred shape: (1024, 6)

Decoder inputs:
  z shape: (1024, 32) (expected: [batch, latent_dim])
  y_batch shape: (1024, 6) (expected: [batch, num_classes])
